In [1]:
import os
import numpy as np
import tensorflow as tf
import rasterio

In [2]:
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D,
    Concatenate, BatchNormalization
)
from tensorflow.keras.models import Model

In [3]:
IMG_DIR = "/Volumes/Windows8_OS/Dataset/Dataset-OG/Images/Oil"
MASK_DIR = "/Volumes/Windows8_OS/Dataset/Dataset-OG/Mask/Oil"


In [4]:
import os

def list_tif_files(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".tif", ".tiff"))
    ])

image_files = list_tif_files(IMG_DIR)
mask_files  = list_tif_files(MASK_DIR)

print("Images:", len(image_files))
print("Masks :", len(mask_files))

Images: 150
Masks : 150


In [5]:
def load_sar_image_and_mask(img_path, mask_path):
    def _read(img_p, mask_p):
        # ---- Image ----
        with rasterio.open(img_p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        # Normalize (paper-consistent)
        vv = np.clip(vv, -35, 5)
        vh = np.clip(vh, -40, 0)
        vv = (vv + 35) / 40
        vh = (vh + 40) / 40

        image = np.stack([vv, vh], axis=-1)

        # ---- Mask ----
        with rasterio.open(mask_p.decode()) as src:
            mask = src.read(1).astype(np.float32)

        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        # 🔑 RESIZE BOTH IMAGE AND MASK (2048 → 512)
        image = tf.image.resize(image, (512, 512), method="bilinear").numpy()
        mask  = tf.image.resize(mask,  (512, 512), method="nearest").numpy()

        return image, mask

    image, mask = tf.numpy_function(
        _read,
        [img_path, mask_path],
        [tf.float32, tf.float32]
    )

    image.set_shape([512, 512, 2])
    mask.set_shape([512, 512, 1])

    return image, mask


In [6]:
def make_segmentation_dataset(img_dir, mask_dir, batch_size=4):
    image_files = sorted([
        os.path.join(img_dir, f)
        for f in os.listdir(img_dir)
        if f.lower().endswith((".tif", ".tiff"))
    ])

    mask_files = sorted([
        os.path.join(mask_dir, f)
        for f in os.listdir(mask_dir)
        if f.lower().endswith((".tif", ".tiff"))
    ])

    ds = tf.data.Dataset.from_tensor_slices((image_files, mask_files))

    ds = ds.map(
        lambda x, y: load_sar_image_and_mask(x, y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.shuffle(len(image_files))
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [7]:
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    return x


def build_unet(input_shape=(512, 512, 2)):
    inputs = Input(input_shape)

    # Encoder
    c1 = conv_block(inputs, 32)
    p1 = MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)
    p4 = MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 512)

    # Decoder
    u1 = UpSampling2D()(bn)
    u1 = Concatenate()([u1, c4])
    c5 = conv_block(u1, 256)

    u2 = UpSampling2D()(c5)
    u2 = Concatenate()([u2, c3])
    c6 = conv_block(u2, 128)

    u3 = UpSampling2D()(c6)
    u3 = Concatenate()([u3, c2])
    c7 = conv_block(u3, 64)

    u4 = UpSampling2D()(c7)
    u4 = Concatenate()([u4, c1])
    c8 = conv_block(u4, 32)

    outputs = Conv2D(1, 1, activation="sigmoid")(c8)

    model = Model(inputs, outputs)
    return model


In [8]:
def dice_loss(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


In [9]:
model = build_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=dice_loss,
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1767801591.938847 3031909 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1767801591.939031 3031909 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 512, 512,  │        608 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512, 512,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 512, 512,  │      9,248 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512, 512,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 256, 256,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 256,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 128, 128,  │    147,584 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_2[

 Total params: 7,858,145 (29.98 MB)

 Trainable params: 7,852,257 (29.95 MB)

 Non-trainable params: 5,888 (23.00 KB)

In [10]:
train_ds = make_segmentation_dataset(
    IMG_DIR,
    MASK_DIR,
    batch_size=8
)

In [ ]:
x, y = next(iter(train_ds))
print(x.shape, y.shape)   # (8, 512, 512, 2) (8, 512, 512, 1)

In [11]:
history = model.fit(train_ds, epochs=10)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


19/19 ━━━━━━━━━━━━━━━━━━━━ 214s 11s/step
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 232s 12s/step
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 210s 11s/step
Epoch 4/10


2026-01-07 21:42:49.963232: W tensorflow/core/framework/op_kernel.cc:1844] UNKNOWN: RasterioIOError: Read failed. See previous exception for details.
rasterio._err.CPLE_AppDefinedError: TIFFReadEncodedStrip:Read error at scanline 4294967295; got 0 bytes, expected 16384


The above exception was the direct cause of the following exception:


rasterio._err.CPLE_AppDefinedError: TIFFReadEncodedStrip() failed.


The above exception was the direct cause of the following exception:


Traceback (most recent call last):

  File "rasterio/_io.pyx", line 969, in rasterio._io.DatasetReaderBase._read

  File "rasterio/_io.pyx", line 199, in rasterio._io.io_multi_band

  File "rasterio/_io.pyx", line 205, in rasterio._io.io_multi_band

  File "rasterio/_err.pyx", line 325, in rasterio._err.StackChecker.exc_wrap_int

rasterio._err.CPLE_AppDefinedError: 00116.tif, band 1: IReadBlock failed at X offset 0, Y offset 1024: TIFFReadEncodedStrip() failed.


The above exception was the direct cause of the f

UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) UNKNOWN:  Error in user-defined function passed to ParallelMapDatasetV2:1 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Shuffle::ParallelMapV2: RasterioIOError: Read failed. See previous exception for details.
rasterio._err.CPLE_AppDefinedError: TIFFReadEncodedStrip:Read error at scanline 4294967295; got 1652 bytes, expected 2048


The above exception was the direct cause of the following exception:


rasterio._err.CPLE_AppDefinedError: TIFFReadEncodedStrip() failed.


The above exception was the direct cause of the following exception:


Traceback (most recent call last):

  File "rasterio/_io.pyx", line 969, in rasterio._io.DatasetReaderBase._read

  File "rasterio/_io.pyx", line 199, in rasterio._io.io_multi_band

  File "raste

In [ ]:
pred = model.predict(image[None, ...])
mask = (pred[0, :, :, 0] > 0.5).astype(np.uint8)